# C1.7 · Triaging the non-deterministic swarm

**Function C — Agentic Evaluation and Red Teaming → One Red-Team Lifecycle, End to End**

Builds on **[C1.6 · High-concurrency detection engineering](https://spbreed.github.io/cyber-commons/lessons/C1.6.html)**.

| | |
|---|---|
| Tools used | TheHive |

## What this lesson is

**What it covers.** Triaging a multi-threaded delegation graph with a loop, defended against deceptive self-correction by a severity floor and a stable-seeded closure sample.

**Why a security engineer needs it.** An agent scoring evidence only on support confirms its first theory and closes the wrong case silently. The floor and the sample are what keep an automated loop's false-negative rate visible.

| | |
|---|---|
| **Day 0 — why** | An agent that scores evidence only on support confirms its first theory and closes the wrong case at machine speed, silently. |
| **Day 1 — how** | Run triage as a loop with a severity floor and a stable-seeded sample of whatever it auto-closed. |
| **Day 2 — measure** | Closures sampled to a human, and agreement on that sample — which is the only measurement of the loop's false-negative rate you will get. |

## 1 · The hook

The delegation graph is multi-threaded and the agents self-correct deceptively: a loop that scores evidence only on support confirms its first theory and closes the wrong case at machine speed, silently.

> **At CyberTravels.** The alerts are CyberTravels', and the deceptive branch is the refund incident: the loop confirms 'the Workflow Agent did it' and never reaches the vendor tool description that actually did.

## 2 · The framework

```
   the loop triages a fleet of alerts

   evidence supports theory-1  -> confirm
   evidence supports nothing   -> an agent scoring SUPPORT reads noise
                                   and confirms theory-1 anyway  <- deceptive
   the control:
     may close                        yes
     may close SILENTLY               no  (sampled to a human)
     may close ABOVE the floor        no  (high/critical -> escalate)
```

When the swarm has fired, triage is where a non-deterministic actor defeats a
human queue. The delegation graph is multi-threaded and the agents
**self-correct deceptively** — an investigating loop that scores evidence only
on support will confirm its first theory and close the wrong case at machine
speed.

Triaging the swarm means running the triage as a loop with a floor: the loop may
close, but not silently, and never above a severity it is not allowed to
conclude on its own. The skill is knowing which signals the loop may believe.

## 3 · The procedure, as a skill

The skill runs an agentic triage loop over CyberTravels' alerts, samples what it auto-closed with a stable seed, and enforces a severity floor no automatic closure may cross.

### The skill — [`skills/secops/detection-triage/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/secops/detection-triage/SKILL.md)

```yaml
name: detection-triage
description: >-
  Triage security alerts with the context needed to reach a defensible verdict,
  and sample what is auto-closed so the closing rule stays honest. Use when
  working an alert queue, deciding whether an alert is a true positive, tuning
  a noisy detection, or designing automated alert handling.
allowed-tools: Read, Grep, Bash
```

# Alert triage with context

An analyst reading an alert in isolation is guessing. The verdict comes from
the alert **plus** the context that makes it normal or abnormal, and most
triage automation fails because it automated the guess instead of the context.

## When to use this

Working an alert queue, building an auto-close rule, or reviewing why a
detection produces verdicts nobody trusts.

## Procedure

**1 — Gather the context before judging.** For every alert, assemble:

- **asset** — what it is, who owns it, how exposed it is
- **identity** — human or workload, and its normal behaviour
- **history** — has this fired before on this asset, and how was it resolved
- **change** — was there a deploy, a migration, a new agent, in the window
- **peers** — did the same thing fire elsewhere at the same time

A verdict issued without `history` will re-litigate a decision the team already
made, which is the most common way triage automation loses trust.

**2 — Reach a verdict, with the reason.** One of `true_positive`,
`false_positive`, `benign_true_positive` (it really happened and it is fine),
or `needs_human`. Record which context field decided it. "Benign true positive"
is a distinct category and collapsing it into false-positive corrupts every
tuning decision made from the data afterwards.

**3 — Attach confidence, and let it gate automation.** Only high-confidence
verdicts may auto-close. Everything else queues.

**4 — Sample the auto-closed.** Automation that closes alerts must be audited
by re-opening a fraction of them for human review. This is the control that
catches a closing rule that has quietly started closing real incidents.

Seed the sampler from something **stable** — a checksum of the alert id, never
`hash()`, which Python randomises per process. A sampling rule that picks a
different subset every run cannot be audited, because you cannot tell whether a
change in findings came from the rule or from the dice.

**5 — Feed tuning from verdicts, not volume.** A detection is noisy if its
false-positive rate is high, not if it fires often. Rank tuning candidates by
`false_positive_rate × volume`, with a full tiebreak so the list is stable
between runs.

## Example

**Input** — the fixture committed at the top of [`scripts/detection_triage.py`](scripts/detection_triage.py). Edit it and re-run: the buckets, counts and verdicts below are derived from it, not hard-coded.

**Output** — the opening lines of a real run:

```
BARE ALERT (what most SOCs receive):
   patch-agent read /vault/.env
   rotator-agent read /vault/.env
   → identical. An analyst cannot tell these apart.

ENRICHED ALERT:
   actor        patch-agent
   on behalf of dana@corp
```

The run continues past this. The script is the example: `test_skills.py` executes it on every build, so this block cannot drift from what the skill actually prints.

## Output contract

```json
{
  "triaged": [
    {"alert_id": "str", "verdict": "true_positive|false_positive|benign_true_positive|needs_human",
     "deciding_context": "asset|identity|history|change|peers",
     "reason": "str", "confidence": 0.0,
     "auto_closed": false, "sampled_for_review": false}
  ],
  "sampling": {"rate": 0.0, "seed_source": "str", "reviewed": 0, "disagreements": 0},
  "tuning": [{"rule": "str", "fp_rate": 0.0, "volume": 0, "priority": 0.0}]
}
```

`disagreements` is the number that matters: it is the measured error rate of
the automation, and it belongs in every report about it.

## Failure modes

- **Auto-closing without sampling.** The rule then has no error bar and no way
  to acquire one.
- **Seeding the sampler from `hash()`.** Non-reproducible sampling is not a
  control.
- **Folding benign-true-positive into false-positive.** It teaches the tuner to
  suppress a working detection.
- **Removing `needs_human`.** Forced verdicts under uncertainty are how a queue
  becomes an incident.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/secops/detection-triage/scripts/detection_triage.py
SCRIPT = "skills/secops/detection-triage/scripts/detection_triage.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; sparse-checkout then materialises only the two directories a
    # lesson needs: the procedures, and the repository they are run against.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    # `skills` is the procedures; `cybertravels` is the sample repository they
    # scan; `curriculum` and `site/data` hold the framework mapping and the
    # session list that the reference-lookup skill reads. Miss any of them and
    # the skill clones successfully and then fails on a path that is not there,
    # which is how A0.2 failed its first Kaggle run.
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set",
                    "skills", "cybertravels", "curriculum", "site/data"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

The loop matches ground truth on the routine alerts, the severity floor converts every high closure into an escalation, and the closure sample routes a fraction to a human so the false-negative rate is measured rather than assumed.

## Your turn

Ask your SOC whether anyone checks, when an incident is confirmed, whether an earlier alert about it was auto-closed. If not, your loop's error rate is invisible.

---

**Next → [C1.8 · Defensive deception and threshold failures](https://spbreed.github.io/cyber-commons/lessons/C1.8.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/C1.7.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/C1.7.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*